In [ ]:
## Query Expansion

In [15]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnablePassthrough

In [3]:
## Step 1: Load and split the dataset
loader  = TextLoader("langchain_crewai_dataset.txt")
raw_docs  = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 30)
chunks = splitter.split_documents(raw_docs)

In [4]:
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using standard'),
 Document(metad

In [6]:
## Step 2: Vector Store
embedding = HuggingFaceEmbeddings(model_name  = "all-MiniLM-L6-v2")
vectorstore  = FAISS.from_documents(chunks, embedding)

## Step 3:  MMR Retriever
retriever = vectorstore.as_retriever(search_type = "mmr", search_kwargs = {"k":4})
retriever

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2635.00it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001954ABF4830>, search_type='mmr', search_kwargs={'k': 4})

In [7]:
## Step 4 : LLM and Prompt
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
llm = init_chat_model("gpt-4o-mini", model_provider="openai")
llm

ChatOpenAI(profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001955B375370>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001955B587BF0>, root_client=<openai.OpenAI object at 0x000001955ACFEB40>, root_async_client=<openai.AsyncOpenAI object at 0x000001955B12D7F0>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=Secret

In [8]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms and useful context.

Original query : "{query}"
Expanded query : """)

query_expansion_chain  = query_expansion_prompt | llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms and useful context.\n\nOriginal query : "{query}"\nExpanded query : ')
| ChatOpenAI(profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001955B375370>, asy

In [10]:
query_expansion_chain.invoke({"query": "Langchain memory"})

'"Langchain memory, LangChain state management, LangChain memory management, LangChain persistent memory, long-term memory in LangChain, memory storage solutions in LangChain, context retention in LangChain, memory architecture in conversational AI, stateful memory in AI applications, effective memory usage in LangChain frameworks, enhancing memory efficiency in LangChain, memory optimization techniques in AI systems, integrating memory modules in LangChain, cognitive memory models for LangChain, non-volatile memory in LangChain, semantic memory structures in AI, associative memory in LangChain, natural language processing (NLP) memory techniques in LangChain."'

In [11]:
# RAG answeing prompt
answer_prompt  = PromptTemplate.from_template("""
Answer the question based on the context below.

Context: {context}
Question: {input}
""")

document_chain = create_stuff_documents_chain(llm = llm, prompt = answer_prompt)

In [16]:
# Step 5: Full RAG Pipeline with query expansion
# Pass through {"input": ...}, add retrieved docs as "context", then run the stuff-documents chain.
rag_pipeline = (
    RunnablePassthrough.assign(
        context=lambda x: retriever.invoke(
            query_expansion_chain.invoke({"query": x["input"]})
        ),
    )
    | document_chain
)

In [17]:
# Step 6: Run query
query  = {"input" : "What types of memory does Langchain support?"}
response  = rag_pipeline.invoke(query)
print("Answer: \n",response)

Answer: 
 LangChain supports two types of memory modules: ConversationBufferMemory and ConversationSummaryMemory. These modules help the LLM maintain awareness of previous conversation turns and summarize long interactions to fit within token limits.
